In [ ]:
import yaml 
from pathlib import Path
from pprint import pprint

import mlflow

from pytorch_pipeline import val
from pytorch_pipeline.train import build_pipeline_dataloaders, build_datasets, get_device
from pytorch_pipeline.utils import resolve_uri, Config, resolve_hardware_profile, get_current_git_branch
from pytorch_pipeline.utils.params import DatasetParams, DataLoadersParams, PathsParams


In [10]:
#Args
test = False
test_fraction = 0.05
seed = 42
model_name = 'cv_pheno_bioclip'
model_version = 1
config_path = Path("/home/etienne/projects/inat-phenology-cv/configs/local.yaml")

In [17]:
# Set up environment specific configs
with open(config_path, "r") as file:
    env_configs = yaml.safe_load(file)
paths_params = PathsParams(**env_configs["paths"])
dataloader_params = DataLoadersParams(**env_configs["dataloader_params"])
dataset_params = DatasetParams(
    **env_configs["dataset_params"], testing_frac=test_fraction)
hardware_profile = resolve_hardware_profile()
configs = Config(
    config_path,
    paths_params=paths_params,
    dataset_params=dataset_params,
    dataloaders_params=dataloader_params,
    hardware_profile=hardware_profile,
    git_branch=get_current_git_branch(),
    )
configs.test = test

In [22]:
# Override max images 
configs.dataloaders_params.max_images = 32

In [13]:
# Load the model
mlflow.set_tracking_uri(resolve_uri())
model_uri = f"models:/{model_name}/{model_version}"
model = mlflow.pytorch.load_model(model_uri)

In [23]:
#Load model, dataset & dataloaders 
device = get_device()
datasets = build_datasets(configs, model, seed= seed)
_, val_loader, _ = build_pipeline_dataloaders(datasets, configs.dataloaders_params, seed=seed)

Running on cuda


In [ ]:
#Run inference
obs_ids, raw_labels, raw_preds = val.execute(model=model, dataloader=val_loader, device=device, as_numpy=True)

In [89]:
import duckdb 

con = duckdb.connect('/home/etienne/projects/inat-phenology-cv/data/cleanlab.duckdb')

con.execute(


    """
        CREATE OR REPLACE TABLE test_set (
        obs_id INTEGER,
        labels DOUBLE[],
        preds DOUBLE[]
    )


    """)

rows = [
    (id, label.tolist(), pred.tolist())
    for i, (id, label, pred) in enumerate(zip(obs_ids, raw_labels, raw_preds))
]
con.executemany("""
    INSERT INTO test_set (obs_id, labels, preds)
    VALUES (?, ?, ?)
""", rows)

